## Streaming Tool-Calling: Anthropic SDK vs OpenAI SDK

| Step | Anthropic SDK | OpenAI SDK |
| --- | --- | --- |
| Setup | `Anthropic()` client + `model` string | `OpenAI()` client (AICredits gateway) + `model` string |
| Message format | Content-block lists (`text`, `tool_use`, `tool_result`) | Flat dicts (`role`, `content`, `tool_calls`) |
| Making the call | `client.beta.messages.stream(**params)`, used as a `with` context manager | `client.chat.completions.create(stream=True, **params)`, iterated directly as a generator |
| Streaming tool args | `input_json` events give partial JSON (needs fine-grained-tool-streaming beta) | `delta.tool_calls[i].function.arguments` gives partial JSON per call `index` |
| Detecting a tool call | `content_block_start` where `content_block.type == "tool_use"` | `delta.tool_calls` present on a chunk |
| Building the final response | `stream.get_final_message()` reconstructs it automatically | Manually accumulated from chunks into an `assistant_message` dict |
| Loop exit condition | `response.stop_reason != "tool_use"` | `not tool_calls` |
| Returning tool results | One user message with a list of `tool_result` blocks | One `tool`-role message per call |
| Forced tool choice | `tool_choice` param; loop breaks after one round-trip | Same `tool_choice` param and behavior |
| Usage | `run_conversation(messages, tools=[...], fine_grained=True)` | `run_conversation(messages, tools=[...])` (no `fine_grained` equivalent) |


In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [ ]:
# Helper functions


# Step: Message format - normalize messages into Anthropic's content-block shape
def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)

# Step: Making the call - stream via client.beta.messages.stream
def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):

    return "\n".join([block.text for block in message.content if block.type == "text"])    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
# Tool definition
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)
save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"


In [ ]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [ ]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                # Incremental assistant text
                if chunk.type == "text":
                    print(chunk.text, end="")

                # Detecting a tool call
                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                # Streaming tool args (requires fine-grained-tool-streaming beta)
                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            # Building the final response
            response = stream.get_final_message()

        add_assistant_message(messages, response)

        # Loop exit condition
        if response.stop_reason != "tool_use":
            break

        # Returning tool results (single combined user message)
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        # Forced tool choice - stop after one round-trip
        if tool_choice:
            break

    return messages

In [ ]:
# Step: Usage
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True
)

## OpenAI SDK Implementation

See the comparison table above for how this section differs from the Anthropic implementation.


In [1]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    base_url="https://api.aicredits.in/v1",
    api_key=os.getenv("AICREDITS_API_KEY"),
)
openai_model = "anthropic/claude-sonnet-4-5"


In [ ]:
# Helper functions


# Step: Message format - flat role/content/tool_calls dicts
def add_user_message(messages, message):
    if isinstance(message, list):
        messages.extend(message)
    else:
        messages.append({"role": "user", "content": message})


def add_assistant_message(messages, message):
    if isinstance(message, dict):
        assistant_message = message
    else:
        assistant_message = {"role": "assistant", "content": message}
    messages.append(assistant_message)

# Step: Making the call - stream via chat.completions.create(stream=True)
def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=None,
    tools=None,
    tool_choice=None,
    # OpenAI has no equivalent of Anthropic's fine-grained-tool-streaming beta
    fine_grained=False,
):
    chat_messages = []
    if system:
        chat_messages.append({"role": "system", "content": system})
    chat_messages.extend(messages)

    params = {
        "model": openai_model,
        "max_tokens": 1000,
        "messages": chat_messages,
        "temperature": temperature,
        "stop": stop_sequences,
        "stream": True,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    return openai_client.chat.completions.create(**params)



In [3]:
# Tool definition

save_article_tool = {
    "type": "function",
    "function": {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "parameters": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    },
}
save_short_article_tool = {
    "type": "function",
    "function": {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "parameters": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    },
}


def save_article(**kwargs):
    return "Article saved!"


In [4]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(tool_calls):
    tool_result_messages = []

    for tool_call in tool_calls:
        tool_input = json.loads(tool_call["function"]["arguments"])
        try:
            tool_output = run_tool(tool_call["function"]["name"], tool_input)
            tool_result_message = {
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "content": json.dumps(tool_output),
            }
        except Exception as e:
            tool_result_message = {
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "content": f"Error: {e}",
            }

        tool_result_messages.append(tool_result_message)

    return tool_result_messages


In [ ]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        stream = chat_stream(
            messages,
            tools=tools,
            tool_choice=tool_choice,
            fine_grained=fine_grained,
        )

        collected_text = []
        tool_calls_by_index = {}

        for chunk in stream:
            delta = chunk.choices[0].delta

            # Incremental assistant text
            if delta.content:
                print(delta.content, end="")
                collected_text.append(delta.content)

            # Detecting a tool call
            if delta.tool_calls:
                for tool_call_delta in delta.tool_calls:
                    index = tool_call_delta.index
                    if index not in tool_calls_by_index:
                        tool_calls_by_index[index] = {
                            "id": tool_call_delta.id,
                            "name": "",
                            "arguments": "",
                        }

                    # Some gateways resend the name on every delta, so only print once
                    if tool_call_delta.function.name and not tool_calls_by_index[index]["name"]:
                        tool_calls_by_index[index]["name"] = tool_call_delta.function.name
                        print(f'\n>>> Tool Call: "{tool_call_delta.function.name}"')

                    # Streaming tool args (arrives per call index)
                    if tool_call_delta.function.arguments:
                        print(tool_call_delta.function.arguments, end="")
                        tool_calls_by_index[index]["arguments"] += tool_call_delta.function.arguments

        print("\n")

        # Building the final response (reconstructed manually from deltas)
        tool_calls = [
            {
                "id": call["id"],
                "type": "function",
                "function": {"name": call["name"], "arguments": call["arguments"]},
            }
            for call in tool_calls_by_index.values()
        ]

        assistant_message = {
            "role": "assistant",
            "content": "".join(collected_text) or None,
        }
        if tool_calls:
            assistant_message["tool_calls"] = tool_calls

        add_assistant_message(messages, assistant_message)

        # Loop exit condition
        if not tool_calls:
            break

        # Returning tool results (one tool-role message per call)
        tool_results = run_tools(tool_calls)
        add_user_message(messages, tool_results)

        # Forced tool choice - stop after one round-trip
        if tool_choice:
            break

    return messages

In [ ]:
# Step: Usage
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_tool],
)


I'll create a fake computer science article for you.
>>> Tool Call: "save_article"
{"abstract": "This paper introduces a novel quantum-inspired algorithm for optimizing neural network architectures through adaptive topology refinement.", "meta": {"word_count":8547,"review":"This paper presents an innovative approach to neural architecture search by leveraging quantum computing principles. The authors propose a hybrid quantum-classical algorithm that dynamically adjusts network topology during training. Experimental results demonstrate a 23% improvement in convergence speed compared to traditional methods on benchmark datasets. The theoretical framework is well-grounded in quantum mechanics and graph theory. However, the computational overhead of the quantum simulation components may limit practical applicability. The paper would benefit from more extensive ablation studies to isolate the contribution of individual components. The writing is generally clear, though some mathematical not

[{'role': 'user',
  'content': 'Create and save a fake computer science article'},
 {'role': 'assistant',
  'content': "I'll create a fake computer science article for you.",
  'tool_calls': [{'id': 'toolu_01B5gUqVFuzirK2qEu1KedsE',
    'type': 'function',
    'function': {'name': 'save_article',
     'arguments': '{"abstract": "This paper introduces a novel quantum-inspired algorithm for optimizing neural network architectures through adaptive topology refinement.", "meta": {"word_count":8547,"review":"This paper presents an innovative approach to neural architecture search by leveraging quantum computing principles. The authors propose a hybrid quantum-classical algorithm that dynamically adjusts network topology during training. Experimental results demonstrate a 23% improvement in convergence speed compared to traditional methods on benchmark datasets. The theoretical framework is well-grounded in quantum mechanics and graph theory. However, the computational overhead of the quantu